# AIA Supervisor Agent — Interactive Notebook

Run each cell in order to build, validate, and test the agent interactively.

**Agents in this system:**

| Agent | Intent | Description |
|---|---|---|
| **Genie** | `simple_kpi`, `multi_domain` | Text-to-SQL via Genie Space |
| **Multi-Tool** | `document_lookup`, `multi_domain` | Vector Search RAG over policy documents |

**Flow:** `classify_intent` → *(optional)* `clarify` → `resolve_assets` → `genie` / `multi_tool` / `all_agents` → `compose_answer`

In [1]:
!pip install "mlflow>=3.1" "databricks-agents>=1.0.0" "pydantic>=2" "langgraph>=0.2" langchain-core databricks-langchain databricks-vectorsearch databricks-sdk databricks-ai-bridge --upgrade

dbutils.library.restartPython()

## 1. Configuration

In [ ]:
import mlflow
import json
import time
import hashlib
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage
from databricks_langchain import ChatDatabricks
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse
from mlflow.entities.span import SpanType
from mlflow.models import set_model

mlflow.langchain.autolog()

CATALOG        = "aia_multi_agent_catalog"
MODEL_ENDPOINT = "databricks-claude-opus-4-6"
VS_INDEX       = f"{CATALOG}.ai_ops.context_index_vs"
VS_ENDPOINT    = "aia_context_index_vs"
DOC_VS_INDEX   = f"{CATALOG}.bronze.policy_documents_vs"
SQL_WAREHOUSE_ID = "4b9b953939869799"

print(f"CATALOG       : {CATALOG}")
print(f"MODEL_ENDPOINT: {MODEL_ENDPOINT}")
print(f"VS_INDEX      : {VS_INDEX}")
print(f"DOC_VS_INDEX  : {DOC_VS_INDEX}")
print(f"WAREHOUSE     : {SQL_WAREHOUSE_ID}")

## 2. SQL Helper

In [ ]:
def _run_sql(sql_statement, max_rows=50):
    """Execute SQL via Databricks SDK Statement Execution API."""
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    response = w.statement_execution.execute_statement(
        warehouse_id=SQL_WAREHOUSE_ID,
        statement=sql_statement,
        wait_timeout="50s",
    )
    if response.status and response.status.state and response.status.state.value == "SUCCEEDED":
        columns = []
        column_meta = []
        try:
            manifest = response.manifest
            if manifest:
                cols_obj = getattr(manifest, 'columns', None)
                if not cols_obj:
                    schema = getattr(manifest, 'schema', None)
                    if schema:
                        cols_obj = getattr(schema, 'columns', None)
                if cols_obj:
                    columns = [c.name for c in cols_obj]
                    column_meta = [{"name": c.name, "type_name": (getattr(c, 'type_text', None) or getattr(c, 'type_name', None) or 'STRING').upper()} for c in cols_obj]
        except (AttributeError, TypeError):
            pass
        if not columns:
            import re
            m = re.match(r'\s*SELECT\s+(.+?)\s+FROM\s', sql_statement, re.IGNORECASE | re.DOTALL)
            if m:
                col_str = m.group(1)
                columns = [c.strip().split('.')[-1].split(' ')[-1] for c in col_str.split(',')]
                column_meta = [{"name": c, "type_name": "STRING"} for c in columns]
        rows = []
        if response.result and response.result.data_array:
            for row in response.result.data_array[:max_rows]:
                if columns and len(columns) == len(row):
                    rows.append(dict(zip(columns, row)))
                else:
                    rows.append({f"col_{i}": v for i, v in enumerate(row)})
        return {"columns": columns, "column_meta": column_meta, "rows": rows, "row_count": len(rows)}
    else:
        error_msg = ""
        if response.status and response.status.error:
            error_msg = response.status.error.message
        raise Exception(f"SQL failed: {error_msg}")

# Smoke test
try:
    result = _run_sql("SELECT current_user() AS user, current_timestamp() AS ts")
    print("SQL helper OK:", result["rows"])
except Exception as e:
    print(f"SQL helper error: {e}")

## 3. Prompt Management (P1)

In [ ]:
_prompt_cache = {}
_prompt_cache_ts = 0


def _load_prompts():
    """Load prompts from ai_ops.agent_instructions. Cached 5 min."""
    global _prompt_cache, _prompt_cache_ts
    if time.time() - _prompt_cache_ts < 300 and _prompt_cache:
        return _prompt_cache
    try:
        result = _run_sql(
            f"SELECT agent_id, scope, base_prompt, overlay_prompt "
            f"FROM {CATALOG}.ai_ops.agent_instructions"
        )
        prompts = {}
        for row in result["rows"]:
            key = f"{row['agent_id']}:{row['scope']}"
            base = row.get("base_prompt", "") or ""
            overlay = row.get("overlay_prompt", "") or ""
            prompts[key] = (base + "\n" + overlay).strip()
        _prompt_cache = prompts
        _prompt_cache_ts = time.time()
    except Exception:
        pass
    return _prompt_cache


def _get_prompt(agent_id, scope, fallback=""):
    prompts = _load_prompts()
    return prompts.get(f"{agent_id}:{scope}", fallback)


prompts = _load_prompts()
print(f"Loaded {len(prompts)} prompts: {list(prompts.keys())}")

## 4. Short-term Memory — Conversation Checkpoints (P0)

In [ ]:
def _save_checkpoint(thread_id, state_data):
    try:
        checkpoint_id = hashlib.md5(f"{thread_id}:{time.time()}".encode()).hexdigest()[:16]
        state_json = json.dumps(state_data, default=str).replace("'", "''")
        safe_json = state_json.replace("\\", "\\\\")
        _run_sql(f"""
            INSERT INTO {CATALOG}.ai_ops.conversations
            (thread_id, checkpoint_id, state_json, created_at)
            VALUES ('{thread_id}', '{checkpoint_id}', '{safe_json}', current_timestamp())
        """)
        return checkpoint_id
    except Exception:
        return None


def _load_checkpoint(thread_id):
    try:
        result = _run_sql(f"""
            SELECT state_json FROM {CATALOG}.ai_ops.conversations
            WHERE thread_id = '{thread_id}'
            ORDER BY created_at DESC LIMIT 1
        """)
        if result["rows"]:
            return json.loads(result["rows"][0]["state_json"])
    except Exception:
        pass
    return None


print("Short-term memory helpers defined.")

## 5. P2 — Long-term Memory & Episodic Memory

In [ ]:
_memory_cache = {}
_memory_cache_ts = 0


def _load_user_memory(user_id):
    global _memory_cache, _memory_cache_ts
    cache_key = f"mem:{user_id}"
    if time.time() - _memory_cache_ts < 60 and cache_key in _memory_cache:
        return _memory_cache[cache_key]
    try:
        result = _run_sql(f"""
            SELECT memory_key, memory_value FROM {CATALOG}.ai_ops.user_memory
            WHERE user_id = '{user_id}'
              AND (expires_at IS NULL OR expires_at > current_timestamp())
            ORDER BY confidence DESC
        """)
        memories = {r["memory_key"]: r["memory_value"] for r in result["rows"]}
        _memory_cache[cache_key] = memories
        _memory_cache_ts = time.time()
        return memories
    except Exception:
        return {}


def _save_user_memory(user_id, memory_key, memory_value, memory_type="preference", confidence=1.0):
    if not user_id or user_id == "anonymous":
        return
    try:
        key_esc = memory_key.replace("'", "''")
        val_esc = memory_value.replace("'", "''")
        _run_sql(f"""
            MERGE INTO {CATALOG}.ai_ops.user_memory AS t
            USING (SELECT '{user_id}' AS user_id, '{key_esc}' AS memory_key,
                   '{val_esc}' AS memory_value, '{memory_type}' AS memory_type,
                   {confidence} AS confidence,
                   current_timestamp() AS created_at, current_timestamp() AS updated_at,
                   NULL AS expires_at) AS s
            ON t.user_id = s.user_id AND t.memory_key = s.memory_key
            WHEN MATCHED THEN UPDATE SET t.memory_value = s.memory_value,
                t.memory_type = s.memory_type, t.confidence = s.confidence, t.updated_at = s.updated_at
            WHEN NOT MATCHED THEN INSERT *
        """)
        global _memory_cache, _memory_cache_ts
        _memory_cache_ts = 0
    except Exception:
        pass


def _extract_and_save_user_facts(user_id, question, answer):
    if not user_id or user_id == "anonymous":
        return
    try:
        _llm = ChatDatabricks(endpoint=MODEL_ENDPOINT, temperature=0)
        prompt = f"""Analyze this conversation and extract any personal facts or preferences the user shared.
User said: {question}\nAssistant responded: {answer}\n
Only extract EXPLICITLY stated facts. Respond in JSON ONLY:
{{\"facts\": [{{\"key\": \"name\", \"value\": \"John\", \"type\": \"fact\"}}]}}
If no facts: {{\"facts\": []}}"""
        response = _llm.invoke([HumanMessage(content=prompt)])
        raw = response.content.strip()
        if "```" in raw:
            raw = raw.split("```")[1].replace("json", "").strip()
        for fact in json.loads(raw).get("facts", []):
            if fact.get("key") and fact.get("value"):
                _save_user_memory(user_id, fact["key"], fact["value"], fact.get("type", "fact"))
    except Exception:
        pass


def _save_episodic_memory(thread_id, user_id, question, intent, domain, agents_used, outcome="success"):
    try:
        episode_id = hashlib.md5(f"{thread_id}:{question}:{time.time()}".encode()).hexdigest()[:20]
        agents_sql = ", ".join([f"'{a}'" for a in agents_used])
        q_esc = question.replace("'", "''")
        _run_sql(f"""
            INSERT INTO {CATALOG}.ai_ops.episodic_memory
            (episode_id, thread_id, user_id, question, intent, domain, agents_used, outcome, created_at)
            VALUES ('{episode_id}', '{thread_id}', '{user_id}', '{q_esc}', '{intent}', '{domain}',
                    ARRAY({agents_sql}), '{outcome}', current_timestamp())
        """)
    except Exception:
        pass


def _get_episodic_lessons(intent, domain, limit=3):
    try:
        result = _run_sql(f"""
            SELECT question, lesson_learned, outcome
            FROM {CATALOG}.ai_ops.episodic_memory
            WHERE intent = '{intent}' AND domain = '{domain}'
              AND lesson_learned IS NOT NULL
            ORDER BY created_at DESC LIMIT {limit}
        """)
        return result["rows"]
    except Exception:
        return []


print("P2 memory helpers defined.")

## 6. P2 — Tool Registry

In [ ]:
_capabilities_cache = []
_capabilities_cache_ts = 0


def _load_agent_capabilities():
    """Load agent capabilities for semantic routing. Cached 5 min."""
    global _capabilities_cache, _capabilities_cache_ts
    if time.time() - _capabilities_cache_ts < 300 and _capabilities_cache:
        return _capabilities_cache
    try:
        result = _run_sql(f"""
            SELECT capability_id, agent_name, capability_name, description,
                   supported_intents, supported_domains, priority
            FROM {CATALOG}.ai_ops.agent_capabilities
            WHERE is_active = true ORDER BY priority ASC
        """)
        _capabilities_cache = result["rows"]
        _capabilities_cache_ts = time.time()
        return _capabilities_cache
    except Exception:
        return []


caps = _load_agent_capabilities()
print(f"Tool registry: {len(caps)} capabilities (0 = table not yet created)")

## 7. Agent State & LLM

In [ ]:
class AgentState(TypedDict):
    messages: list
    user_question: str
    intent: str
    intent_confidence: float
    clarification_message: Optional[str]
    needs_clarification: bool
    resolved_assets: Optional[dict]
    genie_results: Optional[dict]
    multi_tool_results: Optional[dict]
    final_answer: Optional[str]
    warnings: list
    thread_id: Optional[str]
    user_id: Optional[str]


llm = ChatDatabricks(endpoint=MODEL_ENDPOINT, temperature=0.1, max_tokens=2000)
print(f"LLM initialised: {MODEL_ENDPOINT}")

## 8. Node 1 — Classify Intent

In [ ]:
@mlflow.trace(name="classify_intent", span_type=SpanType.CHAIN)
def classify_intent(state):
    question = state["user_question"]
    messages = state.get("messages", [])
    state.setdefault("warnings", [])

    # Resolve short follow-ups using conversation context
    if len(messages) > 1 and len(question.split()) <= 10:
        recent = messages[-(min(len(messages), 6)):-1]
        if recent:
            conv_text = "\n".join([f"{m.get('role','user')}: {m.get('content','')[:300]}" for m in recent])
            try:
                resp = llm.invoke([HumanMessage(content=
                    f"Given this conversation:\n{conv_text}\n\n"
                    f"The user now says: \"{question}\"\n\n"
                    "Rewrite as a complete standalone question. Return ONLY the rewritten question."
                )])
                resolved = resp.content.strip().strip('\"')
                if len(resolved) > len(question):
                    question = resolved
                    state["user_question"] = question
                    state["warnings"].append(f"Follow-up resolved: {resolved[:150]}")
            except Exception as e:
                state["warnings"].append(f"Follow-up resolution failed: {str(e)[:100]}")

    fallback_prompt = """You are an intent classifier for an insurance analytics system.
Classify the following question into exactly ONE category and provide a confidence score (0.0 to 1.0).

Categories:
- \"simple_kpi\": Simple KPI/metric questions (counts, totals, averages, trends by region/product/time)
- \"document_lookup\": Policy terms, coverage details, exclusions, procedures, document search
- \"multi_domain\": Questions spanning multiple domains or requiring data fusion from different sources

Question: {question}

Respond in JSON format ONLY:
{{\"intent\": \"<category>\", \"confidence\": <float>, \"missing_filters\": []}}

If ambiguous or missing key filters (region, time period, product), list them in missing_filters."""

    user_id = state.get("user_id") or "default"
    memory_context = ""
    user_mem = _load_user_memory(user_id)
    if user_mem:
        memory_context = "\nUser preferences: " + "; ".join([f"{k}={v}" for k, v in user_mem.items()])

    prompt_template = _get_prompt("supervisor", "classify_intent", fallback_prompt)
    try:
        prompt = prompt_template.format(question=question)
    except (KeyError, IndexError):
        prompt = fallback_prompt.replace("{question}", question)
    prompt += memory_context

    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    try:
        if "```" in raw:
            raw = raw.split("```")[1].replace("json", "").strip()
        parsed = json.loads(raw)
        intent = parsed.get("intent", "simple_kpi")
        confidence = float(parsed.get("confidence", 0.8))
        missing_filters = parsed.get("missing_filters", [])
    except (json.JSONDecodeError, ValueError):
        intent, confidence, missing_filters = "simple_kpi", 0.5, []

    if intent not in ["simple_kpi", "document_lookup", "multi_domain"]:
        intent = "simple_kpi"

    state["intent"] = intent
    state["intent_confidence"] = confidence
    state["needs_clarification"] = confidence < 0.6 or len(missing_filters) > 0
    if missing_filters:
        state["_missing_filters"] = missing_filters
    return state


# Quick test
_s = {"messages": [{"role": "user", "content": "What is the total claims by region?"}],
      "user_question": "What is the total claims by region?", "intent": "", "intent_confidence": 0.0,
      "needs_clarification": False, "clarification_message": None, "resolved_assets": None,
      "genie_results": None, "multi_tool_results": None, "final_answer": None,
      "warnings": [], "thread_id": None, "user_id": None}
_s = classify_intent(_s)
print(f"Intent: {_s['intent']} | Confidence: {_s['intent_confidence']:.2f} | Needs clarification: {_s['needs_clarification']}")

## 9. Node 2 — Clarify or Disambiguate *(optional)*

In [ ]:
@mlflow.trace(name="clarify_or_disambiguate", span_type=SpanType.CHAIN)
def clarify_or_disambiguate(state):
    """Triggered when intent confidence < 0.6 or required filters are missing."""
    question = state["user_question"]
    messages = state.get("messages", [])
    missing_filters = state.get("_missing_filters", [])

    history_context = ""
    if len(messages) > 1:
        prior = [m for m in messages[:-1] if m.get("role") in ("user", "assistant")]
        if prior:
            history_context = "\n".join([f"{m['role']}: {m['content'][:200]}" for m in prior[-4:]])

    prompt = (
        f"You are helping disambiguate an insurance analytics question.\n\n"
        f"Question: {question}\n"
        f"Detected intent: {state.get('intent','unknown')} (confidence: {state.get('intent_confidence',0):.2f})\n"
        f"Missing filters: {', '.join(missing_filters) if missing_filters else 'none'}\n"
        f"History:\n{history_context or 'No prior context'}\n\n"
        "If context resolves ambiguity set \"resolved\": true, else generate clarification_question.\n"
        "Respond in JSON: {\"resolved\": true/false, \"refined_intent\": \"<intent>\", "
        "\"refined_confidence\": <float>, \"clarification_question\": \"<question>\"}"
    )

    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()
    try:
        if "```" in raw:
            raw = raw.split("```")[1].replace("json", "").strip()
        parsed = json.loads(raw)
        resolved = parsed.get("resolved", True)
        refined_intent = parsed.get("refined_intent", state["intent"])
        refined_confidence = float(parsed.get("refined_confidence", state["intent_confidence"]))
        clarification = parsed.get("clarification_question", "")
    except (json.JSONDecodeError, ValueError):
        resolved, refined_intent, refined_confidence, clarification = True, state["intent"], state["intent_confidence"], ""

    if refined_intent in ["simple_kpi", "document_lookup", "multi_domain"]:
        state["intent"] = refined_intent
        state["intent_confidence"] = refined_confidence
    if not resolved and clarification:
        state["clarification_message"] = clarification
        state["warnings"].append(f"Clarification needed: {clarification}")
    state["needs_clarification"] = False
    return state


print("clarify_or_disambiguate defined.")

## 10. Node 3 — Resolve Assets via Context Index

In [ ]:
def _get_default_assets(intent="simple_kpi"):
    tables_by_intent = {
        "simple_kpi":      [f"{CATALOG}.gold.claims_summary", f"{CATALOG}.gold.policy_performance"],
        "document_lookup": [f"{CATALOG}.bronze.policy_documents"],
        "multi_domain":    [f"{CATALOG}.gold.claims_summary", f"{CATALOG}.gold.agent_performance",
                              f"{CATALOG}.silver.customer_360"],
    }
    return {
        "domain": "claims", "genie_space": None, "metric_views": [],
        "tables": tables_by_intent.get(intent, [f"{CATALOG}.gold.claims_summary"]),
        "document_indexes": [f"{CATALOG}.bronze.policy_documents"],
        "all_assets": [], "endorsement_info": {},
    }


@mlflow.trace(name="resolve_assets_with_context_index", span_type=SpanType.RETRIEVER)
def resolve_assets_with_context_index(state):
    from databricks.sdk import WorkspaceClient
    question = state["user_question"]
    intent   = state.get("intent", "simple_kpi")
    try:
        w = WorkspaceClient()
        results = w.vector_search_indexes.query_index(
            index_name=VS_INDEX,
            columns=["asset_type", "asset_id", "display_name", "text", "domain", "endorsement_level"],
            query_text=question, num_results=10,
        )
        assets = []
        for row in results.result.data_array:
            assets.append({"asset_type": row[0], "asset_id": row[1], "display_name": row[2],
                            "text": row[3], "domain": row[4], "endorsement_level": row[5],
                            "score": float(row[6]) if len(row) > 6 else 0.0})
        # P1: endorsed assets first
        assets.sort(key=lambda a: (0 if a.get("endorsement_level") == "endorsed" else 1, -a.get("score", 0)))
        domain_counts = {}
        for a in assets[:5]:
            d = a.get("domain", "unknown")
            domain_counts[d] = domain_counts.get(d, 0) + 1
        primary_domain = max(domain_counts, key=domain_counts.get) if domain_counts else "claims"
        genie_spaces = [a for a in assets if a["asset_type"] == "genie_space"]
        metric_views = [a for a in assets if a["asset_type"] == "metric_view"]
        tables       = [a for a in assets if a["asset_type"] == "table"]
        doc_indexes  = [a for a in assets if a["asset_type"] == "document_index"]
        state["resolved_assets"] = {
            "domain": primary_domain,
            "genie_space": genie_spaces[0]["asset_id"] if genie_spaces else None,
            "metric_views": [m["asset_id"] for m in metric_views],
            "tables": [t["asset_id"] for t in tables],
            "document_indexes": [d["asset_id"] for d in doc_indexes],
            "dashboards": [],
            "all_assets": assets,
            "endorsement_info": {a["asset_id"]: a["endorsement_level"] for a in assets},
        }
        print(f"  Domain: {primary_domain} | Genie: {state['resolved_assets']['genie_space']}")
        print(f"  Tables: {state['resolved_assets']['tables']}")
    except Exception as e:
        state["resolved_assets"] = _get_default_assets(intent)
        state["warnings"].append("Context Index fallback — using rule-based asset resolution")
        print(f"  Fallback assets. Error: {e}")
    return state


# Test
_s = resolve_assets_with_context_index(dict(_s))
print("Assets resolved:", list(_s["resolved_assets"].keys()))

## 11. Context Index Helpers

In [ ]:
def _scoped_context_index_lookup(query_text, domain, asset_types=None, num_results=5):
    """Worker-scoped lookup — only returns assets within the Supervisor's resolved domain."""
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        results = w.vector_search_indexes.query_index(
            index_name=VS_INDEX,
            columns=["asset_type", "asset_id", "display_name", "text", "domain", "endorsement_level"],
            query_text=query_text, num_results=num_results,
        )
        assets = []
        for row in results.result.data_array:
            asset = {"asset_type": row[0], "asset_id": row[1], "display_name": row[2],
                     "text": row[3], "domain": row[4], "endorsement_level": row[5],
                     "score": float(row[6]) if len(row) > 6 else 0.0}
            if asset.get("domain", "").lower() == domain.lower():
                if asset_types is None or asset.get("asset_type") in asset_types:
                    assets.append(asset)
        assets.sort(key=lambda a: (0 if a.get("endorsement_level") == "endorsed" else 1, -a.get("score", 0)))
        return assets
    except Exception:
        return []


def _record_asset_feedback(agent_name, domain, feedback_type, details, state):
    """Record when a worker agent finds a missing or useful asset."""
    try:
        user_id     = state.get("user_id", "default")
        question    = state.get("user_question", "")[:200].replace("'", "''")
        details_esc = str(details)[:500].replace("'", "''")
        _run_sql(f"""
            INSERT INTO {CATALOG}.ai_ops.asset_feedback
            (agent_name, domain, feedback_type, details, user_question, user_id, created_at)
            VALUES ('{agent_name}', '{domain}', '{feedback_type}', '{details_esc}',
                    '{question}', '{user_id}', current_timestamp())
        """)
    except Exception:
        pass


print("Context index helpers defined.")

## 12. Node 4 — Genie Agent

In [ ]:
@mlflow.trace(name="route_to_genie", span_type=SpanType.TOOL)
def route_to_genie(state):
    from databricks.sdk import WorkspaceClient
    question = state["user_question"]
    w = WorkspaceClient()

    # Read Genie Space ID from config table, fallback to hardcoded
    space_id = None
    try:
        cfg = _run_sql(f"SELECT config_value FROM {CATALOG}.ai_ops.agent_config WHERE config_key = 'bajaj_genie_space_id'")
        space_id = cfg["rows"][0]["config_value"] if cfg["rows"] else None
    except Exception as e:
        state["warnings"].append(f"Config table read failed: {str(e)[:100]}")
    if not space_id:
        space_id = "01f0d6ff25da1f229950bb97c1ec974c"

    print(f"  Genie Space: {space_id}")
    try:
        conv = w.genie.start_conversation(space_id=space_id, content=question)
        msg  = None
        for _ in range(30):
            msg = w.genie.get_message(space_id=space_id,
                                      conversation_id=conv.conversation_id,
                                      message_id=conv.message_id)
            if msg.status and msg.status.value in ["COMPLETED", "FAILED"]:
                break
            time.sleep(2)

        if msg and msg.status and msg.status.value == "COMPLETED":
            sql_query, result_data = None, None
            if msg.attachments:
                for att in msg.attachments:
                    if att.query and att.query.query:
                        sql_query = att.query.query
                    if att.text and att.text.content:
                        result_data = att.text.content
            state["genie_results"] = {
                "space_id": space_id, "sql": sql_query,
                "result_summary": result_data or "No result text", "status": "success"
            }
            print(f"  Genie: success | SQL: {(sql_query or '')[:80]}")
        else:
            state["genie_results"] = {"error": f"Genie status: {msg.status if msg else 'N/A'}", "status": "failed"}
            state["warnings"].append("Genie query did not complete")
            print(f"  Genie: failed")
    except Exception as e:
        state["genie_results"] = {"error": str(e)[:200], "status": "failed"}
        state["warnings"].append(f"Genie error: {str(e)[:100]}")
        print(f"  Genie exception: {e}")

    # Scoped enrichment if Genie failed
    genie_res = state.get("genie_results", {})
    domain = state.get("resolved_assets", {}).get("domain", "claims")
    if genie_res.get("status") != "success" or not genie_res.get("sql"):
        extra = _scoped_context_index_lookup(question, domain,
                                              asset_types=["metric_view", "genie_space", "table"], num_results=3)
        if extra:
            genie_res["ci_enrichment"] = [{"asset_id": a["asset_id"], "display_name": a["display_name"],
                                             "asset_type": a["asset_type"]} for a in extra]
        if genie_res.get("status") != "success":
            _record_asset_feedback("genie", domain, "genie_query_failed",
                                   f"Genie could not answer: {question[:150]}", state)
    return state


# Test
_s = route_to_genie(dict(_s))
print(f"Genie status: {_s.get('genie_results', {}).get('status', 'N/A')}")

## 13. Node 5 — Multi-Tool Agent *(Vector Search RAG)*

In [ ]:
@mlflow.trace(name="route_to_multi_tool", span_type=SpanType.TOOL)
def route_to_multi_tool(state):
    """Vector Search RAG over policy documents."""
    question = state["user_question"]
    results  = {}
    print(f"  RAG query: {question[:80]}")
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        vs_results = w.vector_search_indexes.query_index(
            index_name=DOC_VS_INDEX,
            columns=["document_id", "title", "content", "document_type", "category"],
            query_text=question, num_results=5,
        )
        docs = [
            {"document_id": r[0], "title": r[1], "content": r[2], "document_type": r[3], "category": r[4]}
            for r in vs_results.result.data_array
        ]
        results["docs"]   = docs
        results["status"] = "success"
        print(f"  RAG: {len(docs)} documents found")
    except Exception as e:
        results["docs"]  = []
        results["error"] = str(e)[:200]
        state["warnings"].append(f"Document retrieval failed: {str(e)[:100]}")
        print(f"  RAG error: {e}")
    state["multi_tool_results"] = results
    return state


# Test with a document question
_rag_s = {"messages": [{"role": "user", "content": "What does the AIA health plan cover?"}],
          "user_question": "What does the AIA health plan cover?",
          "intent": "document_lookup", "intent_confidence": 0.9, "needs_clarification": False,
          "clarification_message": None, "resolved_assets": _get_default_assets("document_lookup"),
          "genie_results": None, "multi_tool_results": None, "final_answer": None,
          "warnings": [], "thread_id": None, "user_id": None}
_rag_s = route_to_multi_tool(_rag_s)
_docs = _rag_s["multi_tool_results"].get("docs", [])
print(f"\nDocuments retrieved: {len(_docs)}")
for d in _docs[:2]:
    print(f"  [{d['document_type']}] {d['title']}: {d['content'][:100]}...")

## 14. Node 6 — Compose Final Answer

In [ ]:
@mlflow.trace(name="compose_answer", span_type=SpanType.CHAIN)
def compose_answer(state):
    question   = state["user_question"]
    intent     = state.get("intent", "unknown")
    assets     = state.get("resolved_assets", {})
    genie      = state.get("genie_results", {})
    multi_tool = state.get("multi_tool_results", {})
    clarification = state.get("clarification_message", "")

    context_parts = []

    if genie and genie.get("status") == "success":
        context_parts.append(f"**Genie Agent:** SQL: {genie.get('sql','N/A')}\nResult: {genie.get('result_summary','N/A')}")

    if multi_tool and multi_tool.get("docs"):
        doc_text = "\n".join([f"- {d['title']}: {d['content'][:200]}..." for d in multi_tool["docs"][:3]])
        context_parts.append(f"**Documents:**\n{doc_text}")

    # P2: Episodic lessons
    domain  = assets.get("domain", "unknown") if assets else "unknown"
    lessons = _get_episodic_lessons(intent, domain)
    if lessons:
        lesson_text = "\n".join([f"- [{l.get('outcome','?')}] {l.get('lesson_learned','')}" for l in lessons if l.get("lesson_learned")])
        if lesson_text:
            context_parts.append(f"**Past experience (internal):**\n{lesson_text}")

    # P2: User memory
    user_id  = state.get("user_id") or "default"
    user_mem = _load_user_memory(user_id)
    if user_mem:
        pref_parts = []
        for k in ("name", "preferred_view", "response_length", "role",
                  "preferred_region", "preferred_domain", "display_currency",
                  "default_time_range", "expertise_level", "team_size"):
            if user_mem.get(k):
                pref_parts.append(f"{k}: {user_mem[k]}")
        if pref_parts:
            context_parts.append(f"**User preferences:** {', '.join(pref_parts)}")

    context = "\n\n".join(context_parts) if context_parts else "No agent results available."

    warning_text = (
        "\n\nCRITICAL RULE: NEVER include a 'Warnings & Limitations' section, warnings, caveats, "
        "disclaimers, or notes about data limitations. These are handled separately by the UI."
    )
    clarification_text = f"\n\nAlso note this clarification: {clarification}" if clarification else ""

    compose_prompt = (
        f"Answer this question naturally and conversationally: {question}\n\n"
        f"Agent Results:\n{context}\n\n"
        "Instructions:\n"
        "- Answer like a knowledgeable insurance colleague — direct, natural, and helpful.\n"
        "- Lead with the key insight, not an introduction.\n"
        "- Include specific numbers naturally in your sentences.\n"
        "- Use markdown when it helps readability.\n"
        f"- Address user by name if known from User preferences.{warning_text}{clarification_text}"
    )

    response = llm.invoke([HumanMessage(content=compose_prompt)])
    state["final_answer"] = response.content
    return state


print("compose_answer defined.")

## 15. Run All Agents & Routing Logic

In [ ]:
@mlflow.trace(name="run_all_agents", span_type=SpanType.CHAIN)
def run_all_agents(state):
    """For multi_domain: run Genie + Multi-Tool in parallel."""
    import copy
    from concurrent.futures import ThreadPoolExecutor

    genie_state      = copy.deepcopy(state)
    multi_tool_state = copy.deepcopy(state)

    with ThreadPoolExecutor(max_workers=2) as executor:
        genie_f      = executor.submit(route_to_genie, genie_state)
        multi_tool_f = executor.submit(route_to_multi_tool, multi_tool_state)
        genie_result      = genie_f.result()
        multi_tool_result = multi_tool_f.result()

    state["genie_results"]      = genie_result.get("genie_results")
    state["multi_tool_results"] = multi_tool_result.get("multi_tool_results")
    for w in genie_result.get("warnings", []):
        if w not in state["warnings"]:
            state["warnings"].append(w)
    for w in multi_tool_result.get("warnings", []):
        if w not in state["warnings"]:
            state["warnings"].append(w)
    return state


def should_clarify(state):
    return "clarify" if state.get("needs_clarification", False) else "resolve"


def route_by_intent(state):
    intent    = state.get("intent", "simple_kpi")
    assets    = state.get("resolved_assets", {})
    has_genie = bool(assets.get("genie_space"))

    # P2: Tool registry
    capabilities = _load_agent_capabilities()
    if capabilities:
        matching = []
        for cap in capabilities:
            cap_intents = cap.get("supported_intents", "[]")
            if isinstance(cap_intents, str):
                try:
                    cap_intents = json.loads(cap_intents.replace("'", '"'))
                except Exception:
                    cap_intents = []
            if intent in cap_intents:
                matching.append((cap["agent_name"], int(cap.get("priority", 100))))
        if matching:
            matching.sort(key=lambda x: x[1])
            best = matching[0][0]
            agent_map = {"genie": "genie", "multi_tool": "multi_tool"}
            if best in agent_map and intent != "multi_domain":
                return agent_map[best]

    # Fallback
    if intent == "multi_domain":    return "all_agents"
    if intent == "document_lookup": return "multi_tool"
    if intent == "simple_kpi":      return "genie" if has_genie else "multi_tool"
    return "multi_tool"


print("run_all_agents and routing logic defined.")

## 16. Build & Compile LangGraph

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("classify_intent",        classify_intent)
workflow.add_node("clarify_or_disambiguate", clarify_or_disambiguate)
workflow.add_node("resolve_assets",          resolve_assets_with_context_index)
workflow.add_node("genie",                   route_to_genie)
workflow.add_node("multi_tool",              route_to_multi_tool)
workflow.add_node("all_agents",              run_all_agents)
workflow.add_node("compose_answer",          compose_answer)

workflow.add_edge(START, "classify_intent")
workflow.add_conditional_edges(
    "classify_intent", should_clarify,
    {"clarify": "clarify_or_disambiguate", "resolve": "resolve_assets"},
)
workflow.add_edge("clarify_or_disambiguate", "resolve_assets")
workflow.add_conditional_edges(
    "resolve_assets", route_by_intent,
    {"genie": "genie", "multi_tool": "multi_tool", "all_agents": "all_agents"},
)
workflow.add_edge("genie",      "compose_answer")
workflow.add_edge("multi_tool", "compose_answer")
workflow.add_edge("all_agents", "compose_answer")
workflow.add_edge("compose_answer", END)

graph = workflow.compile()
print("Graph compiled. Nodes:", list(graph.nodes))

## 17. End-to-End Tests

In [ ]:
def _run_test(question):
    print(f"\n{'='*70}")
    print(f"Q: {question}")
    print(f"{'='*70}")
    state = {
        "messages": [{"role": "user", "content": question}],
        "user_question": question, "intent": "", "intent_confidence": 0.0,
        "clarification_message": None, "needs_clarification": False,
        "resolved_assets": None, "genie_results": None, "multi_tool_results": None,
        "final_answer": None, "warnings": [], "thread_id": None, "user_id": None,
    }
    result = graph.invoke(state)
    print(f"\nIntent   : {result['intent']} ({result.get('intent_confidence',0):.0%})")
    print(f"Genie    : {'yes' if result.get('genie_results') else 'no'}")
    print(f"MultiTool: {'yes' if result.get('multi_tool_results') else 'no'}")
    if result.get("warnings"):
        print(f"Warnings : {result['warnings']}")
    print(f"\nAnswer:\n{result.get('final_answer','N/A')[:600]}")
    return result


# Test 1: KPI question → Genie
_run_test("What is the total number of claims by region?")

In [ ]:
# Test 2: Document lookup → Multi-Tool RAG
_run_test("What does the AIA Health plan cover for hospitalisation?")

In [ ]:
# Test 3: Multi-domain → both agents in parallel
_run_test("How do our top agents\' churn rates compare with total claims volume?")

## 18. MLflow ResponsesAgent Wrapper

In [ ]:
class SupervisorResponsesAgent(ResponsesAgent):

    @mlflow.trace(span_type=SpanType.AGENT)
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        user_message = None
        all_messages = []
        for item in request.input:
            role    = getattr(item, "role", "user")
            content = getattr(item, "content", "")
            if isinstance(content, list):
                parts = []
                for p in content:
                    if hasattr(p, "text"):              parts.append(p.text)
                    elif isinstance(p, dict) and "text" in p: parts.append(p["text"])
                    elif isinstance(p, str):             parts.append(p)
                content = " ".join(parts)
            elif not isinstance(content, str):
                content = str(content) if content else ""
            all_messages.append({"role": role, "content": content})
            if role == "user":
                user_message = content

        if not user_message:
            return ResponsesAgentResponse(
                output=[self.create_text_output_item(text="Please ask a question.", id="msg_empty")])

        custom_inputs = {}
        if hasattr(request, "custom_inputs") and request.custom_inputs:
            custom_inputs = request.custom_inputs if isinstance(request.custom_inputs, dict) else {}
        thread_id = custom_inputs.get("thread_id")
        user_id   = custom_inputs.get("user_id")

        if thread_id and len(all_messages) <= 1:
            prior_state = _load_checkpoint(thread_id)
            if prior_state and prior_state.get("messages"):
                all_messages = prior_state["messages"] + all_messages

        initial_state = {
            "messages": all_messages, "user_question": user_message,
            "intent": "", "intent_confidence": 0.0,
            "clarification_message": None, "needs_clarification": False,
            "resolved_assets": None, "genie_results": None, "multi_tool_results": None,
            "final_answer": None, "warnings": [], "thread_id": thread_id, "user_id": user_id,
        }
        result = graph.invoke(initial_state)
        answer = result.get("final_answer", "I was unable to process your question.")

        checkpoint_id = None
        if thread_id:
            checkpoint_id = _save_checkpoint(thread_id, {
                "messages": all_messages + [{"role": "assistant", "content": answer}],
                "intent":   result.get("intent"),
                "domain":   result.get("resolved_assets", {}).get("domain") if result.get("resolved_assets") else None,
            })

        agents_used = (["genie"]      if result.get("genie_results")      else []) + \
                      (["multi_tool"] if result.get("multi_tool_results") else [])
        ep_domain   = result.get("resolved_assets", {}).get("domain", "unknown") if result.get("resolved_assets") else "unknown"
        has_errors  = any(r.get("status") == "failed" for r in
                          [result.get("genie_results", {}), result.get("multi_tool_results", {})] if isinstance(r, dict))
        _save_episodic_memory(thread_id or "anonymous", user_id or "anonymous",
                              user_message, result.get("intent", "unknown"),
                              ep_domain, agents_used, "failed" if has_errors and not answer else "success")

        if result.get("intent", "") in ("conversational", "unknown", "greeting"):
            _extract_and_save_user_facts(user_id or "default", user_message, answer)

        nodes_executed = [n for n in [
            "classify_intent",
            "clarify_or_disambiguate" if result.get("clarification_message") else None,
            "resolve_assets",
            "genie"      if result.get("genie_results")      else None,
            "multi_tool" if result.get("multi_tool_results") else None,
            "compose_answer",
        ] if n is not None]

        resolved       = result.get("resolved_assets", {}) or {}
        agent_details  = {}
        if result.get("genie_results"):
            gr = result["genie_results"]
            agent_details["genie"] = {"status": gr.get("status"), "sql": (gr.get("sql") or "")[:120]}
        if result.get("multi_tool_results"):
            mt = result["multi_tool_results"]
            agent_details["multi_tool"] = {"docs_found": len(mt.get("docs", []))}

        custom_outputs = {
            "intent":               result.get("intent", "unknown"),
            "intent_confidence":    result.get("intent_confidence", 0.0),
            "domain":              resolved.get("domain", "unknown"),
            "resolved_tables":     [t if isinstance(t, str) else t.get("asset_id", "") for t in resolved.get("tables", [])][:5],
            "resolved_genie_spaces": [g if isinstance(g, str) else g.get("asset_id", "") for g in resolved.get("genie_spaces", [])][:3],
            "warnings":             result.get("warnings", []),
            "clarification":        result.get("clarification_message"),
            "nodes_executed":       nodes_executed,
            "agent_details":        agent_details,
            "thread_id":            thread_id,
            "checkpoint_id":        checkpoint_id,
        }
        return ResponsesAgentResponse(
            output=[
                self.create_text_output_item(text=answer,                    id="msg_answer"),
                self.create_text_output_item(text=json.dumps(custom_outputs), id="msg_metadata"),
            ],
            custom_outputs=custom_outputs,
        )


agent = SupervisorResponsesAgent()
set_model(agent)
print("SupervisorResponsesAgent registered with set_model().")

## 19. Log & Register with MLflow

In [ ]:
import requests, base64

TOKEN    = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
HOST     = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
WS_PATH  = "/Users/sourav.banerjee@databricks.com/aia-multi-agent/agents/agent_code"
agent_file_path = "/tmp/agent_code.py"

resp = requests.get(
    f"{HOST}/api/2.0/workspace/export",
    headers={"Authorization": f"Bearer {TOKEN}"},
    params={"path": WS_PATH, "format": "SOURCE", "direct_download": "false"}
)
resp.raise_for_status()
agent_code = base64.b64decode(resp.json()["content"]).decode("utf-8")
with open(agent_file_path, "w") as f:
    f.write(agent_code)
print(f"agent_code.py written to {agent_file_path} ({len(agent_code)} chars)")

In [ ]:
mlflow.set_experiment(
    f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/aia_supervisor_agent"
)

with mlflow.start_run(run_name="supervisor_agent_v1"):
    model_info = mlflow.pyfunc.log_model(
        name="supervisor_agent",
        python_model=agent_file_path,
        pip_requirements=[
            "mlflow>=3.1",
            "databricks-agents>=1.0.0",
            "pydantic>=2",
            "langgraph>=0.2",
            "langchain-core",
            "databricks-langchain",
            "databricks-vectorsearch",
            "databricks-sdk",
            "databricks-ai-bridge",
        ],
        registered_model_name=f"{CATALOG}.ai_ops.supervisor_agent",
    )
    print(f"Logged model: {model_info.model_uri}")